## Setup
This notebook uses the local `.venv` virtual environment.  
**Select kernel:** `Python (Spreadley)` in the top-right kernel picker before running.

In [5]:
import pandas as pd

In [6]:
df = pd.read_csv('pilot_transcripts.csv', parse_dates=['timestamp'])
df.head()

,session_id,turn_number,speaker,message,sentiment,sentiment_score,timestamp,initial_mood,final_mood
0,6abc1e3b-0f8a-423d-aed6-6b84640dcda1,1,User,Great team and exciting new things coming up!,positive,90.0,2026-03-23 15:38:35.718+00,NaN,NaN
1,6abc1e3b-0f8a-423d-aed6-6b84640dcda1,2,Bot,"When you say it's a 'great team,' what does th...",NaN,NaN,2026-03-23 15:38:35.718+00,NaN,NaN
2,6abc1e3b-0f8a-423d-aed6-6b84640dcda1,3,User,supportive and super friendly atmosphere,positive,90.0,2026-03-23 15:39:02.48+00,NaN,NaN
3,6abc1e3b-0f8a-423d-aed6-6b84640dcda1,4,Bot,Can you think of a specific time recently when...,NaN,NaN,2026-03-23 15:39:02.48+00,NaN,NaN
4,6abc1e3b-0f8a-423d-aed6-6b84640dcda1,5,User,A colleague tested something for me while I wa...,positive,95.0,2026-03-23 15:40:04.987+00,NaN,NaN


In [12]:
ts = pd.to_datetime(df['timestamp'], utc=True, format='ISO8601')
print(f"Shape:          {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Sessions:       {df['session_id'].nunique()} pilot conversations")
print(f"Date range:     {ts.min().date()} → {ts.max().date()}")
print(f"Turns (each):   {df['speaker'].value_counts()['User']} user  |  {df['speaker'].value_counts()['Bot']} bot")
print()
print("User sentiment breakdown:")
print(df.dropna(subset=['sentiment'])['sentiment'].value_counts().to_string())
print()
print(f"Sentiment score: {df['sentiment_score'].min():.0f} – {df['sentiment_score'].max():.0f}  (mean {df['sentiment_score'].mean():.1f})")
print()
print("Columns with all nulls:", df.columns[df.isnull().all()].tolist())

Shape:          98 rows × 9 columns
Sessions:       5 pilot conversations
Date range:     2026-03-23 → 2026-03-25
Turns (each):   49 user  |  49 bot

User sentiment breakdown:
sentiment
positive    28
neutral     16
negative     5

Sentiment score: 20 – 95  (mean 68.0)

Columns with all nulls: ['initial_mood', 'final_mood']


In [9]:
# ── C0: Config ───────────────────────────────────────────────────────────────
# Edit these values before running the pipeline.

# LLM
LLM_PROVIDER = "anthropic"
LLM_MODEL    = "claude-haiku-4-5-20251001"

# File paths
CSV_PATH   = "pilot_transcripts.csv"
KEYS_ENV   = "keys.env"
OUTPUT_DIR = "pipeline_output"

# Coding ranges  (min, max)
L1_CODES_RANGE = (1, 10)   # open codes per Q&A pair
L2_CODES_RANGE = (20, 30)  # consolidated codes per interview
L3_CODES_RANGE = (40, 80)  # consolidated codes across all employees
CLUSTERS_RANGE = (7, 12)   # thematic clusters

print("Config loaded:")
print(f"  Model:    {LLM_PROVIDER} / {LLM_MODEL}")
print(f"  CSV:      {CSV_PATH}")
print(f"  Output:   {OUTPUT_DIR}/")
print(f"  L1 range: {L1_CODES_RANGE}")
print(f"  L2 range: {L2_CODES_RANGE}")
print(f"  L3 range: {L3_CODES_RANGE}")
print(f"  Clusters: {CLUSTERS_RANGE}")

Config loaded:
  Model:    anthropic / claude-haiku-4-5-20251001
  CSV:      pilot_transcripts.csv
  Output:   pipeline_output/
  L1 range: (1, 10)
  L2 range: (20, 30)
  L3 range: (40, 80)
  Clusters: (7, 12)


In [13]:
# ── C1: Imports & env ────────────────────────────────────────────────────────
import re
import json
import os
from collections import defaultdict
from dotenv import load_dotenv

load_dotenv(KEYS_ENV)

loaded_keys = [k for k in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY"] if os.getenv(k)]
print(f"Env loaded from '{KEYS_ENV}'")
print(f"Keys found: {loaded_keys if loaded_keys else 'none — check keys.env'}")

Env loaded from 'keys.env'
Keys found: ['ANTHROPIC_API_KEY']


In [ ]:
# ── C2: LLM client ───────────────────────────────────────────────────────────
# To swap providers: change LLM_PROVIDER in C0 and uncomment the matching branch here.
# verify=False   — bypasses corporate TLS inspection (API key still authenticates).
# trust_env=False — ignores any HTTPS_PROXY / HTTP_PROXY env vars; connect directly.

def call_llm(prompt: str, system: str = "You are a qualitative research assistant.") -> str:
    if LLM_PROVIDER == "anthropic":
        import anthropic, httpx
        client = anthropic.Anthropic(
            http_client=httpx.Client(verify=False, trust_env=False)
        )
        resp = client.messages.create(
            model=LLM_MODEL, max_tokens=2048,
            system=system,
            messages=[{"role": "user", "content": prompt}]
        )
        return resp.content[0].text
    # elif LLM_PROVIDER == "openai":
    #     import openai, httpx
    #     client = openai.OpenAI(http_client=httpx.Client(verify=False, trust_env=False))
    #     resp = client.chat.completions.create(
    #         model=LLM_MODEL, max_tokens=2048,
    #         messages=[{"role": "system", "content": system},
    #                   {"role": "user",   "content": prompt}]
    #     )
    #     return resp.choices[0].message.content
    else:
        raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER!r}")

print(f"LLM client ready: {LLM_PROVIDER} / {LLM_MODEL}")

In [15]:
# ── C3: Data ingestion ───────────────────────────────────────────────────────
# CSV → standardised `interviews` list.  Edit this cell to adapt to future input formats.

import pandas as pd

def load_interviews(csv_path: str) -> list:
    df = pd.read_csv(csv_path)

    required = {"session_id", "turn_number", "speaker", "message"}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(f"CSV missing columns: {missing}")
    if df["session_id"].isnull().any():
        raise ValueError("CSV contains null session_id values")
    if not (df["turn_number"] > 0).all():
        raise ValueError("turn_number must be positive integers")

    interviews = []
    for interview_id, group in df.groupby("session_id"):
        group = group.sort_values("turn_number")

        bot_msgs = {
            int(row["turn_number"]): row["message"]
            for _, row in group.iterrows()
            if row["speaker"] == "Bot"
        }

        qa_pairs = []
        for _, row in group.iterrows():
            if row["speaker"] != "User":
                continue
            turn     = int(row["turn_number"])
            question = bot_msgs.get(turn - 1, "[initial mood opener]")
            qa_pairs.append({
                "turn_number": turn,
                "question":    question,
                "answer":      str(row["message"])
            })

        if not qa_pairs:
            raise ValueError(f"Interview {interview_id} has no User turns")
        interviews.append({"interview_id": interview_id, "qa_pairs": qa_pairs})

    if not interviews:
        raise ValueError("No interviews loaded — check CSV_PATH")
    return interviews

interviews = load_interviews(CSV_PATH)

print(f"Loaded {len(interviews)} interviews\n")
for iv in interviews:
    print(f"  {iv['interview_id'][:8]}...  {len(iv['qa_pairs'])} Q&A pairs")

print("\nSample (first interview, first 2 pairs):")
for qa in interviews[0]["qa_pairs"][:2]:
    print(f"  [t{qa['turn_number']}] Q: {qa['question'][:65]}")
    print(f"         A: {qa['answer'][:65]}")
    print()

Loaded 5 interviews

  6abc1e3b...  9 Q&A pairs
  857cc662...  9 Q&A pairs
  86206459...  11 Q&A pairs
  bc434cbe...  10 Q&A pairs
  f2e9c94a...  10 Q&A pairs

Sample (first interview, first 2 pairs):
  [t1] Q: [initial mood opener]
         A: Great team and exciting new things coming up!

  [t3] Q: When you say it's a 'great team,' what does that look like in you
         A: supportive and super friendly atmosphere



In [16]:
# ── C4: Anonymizer ───────────────────────────────────────────────────────────

_PII_PATTERNS = [
    (r'\b[A-Z][a-z]+ [A-Z][a-z]+\b',                         "[NAME]",  0),
    (r'\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Z]{2,}\b',  "[EMAIL]", re.IGNORECASE),
    (r'\b(?:\+?\d[\d\s\-().]{7,}\d)\b',                      "[PHONE]", 0),
]

def anonymize(text: str) -> str:
    for pattern, replacement, flags in _PII_PATTERNS:
        text = re.sub(pattern, replacement, text, flags=flags)
    return text

sample_raw  = interviews[0]["qa_pairs"][0]["answer"]
sample_anon = anonymize(sample_raw)
print("Anonymizer ready.")
print(f"  Before: {sample_raw[:80]}")
print(f"  After:  {sample_anon[:80]}")

Anonymizer ready.
  Before: Great team and exciting new things coming up!
  After:  Great team and exciting new things coming up!


In [17]:
# ── C5: DB init ──────────────────────────────────────────────────────────────
# PK = interview_question_id (e.g. "6abc1e3b_t1").  Shape is fixed after this cell.

db = {}
for iv in interviews:
    prefix = iv["interview_id"][:8]
    for qa in iv["qa_pairs"]:
        iq_id = f"{prefix}_t{qa['turn_number']}"
        db[iq_id] = {
            "interview_id":      iv["interview_id"],
            "turn_number":       qa["turn_number"],
            "question":          qa["question"],
            "anonymised_answer": anonymize(qa["answer"]),
            "l1_codes":          []
        }

print(f"DB initialised: {len(db)} entries")
sample_key = next(iter(db))
print(f"\nSample entry  ({sample_key}):")
for k, v in db[sample_key].items():
    print(f"  {k}: {str(v)[:80]!r}")

DB initialised: 49 entries

Sample entry  (6abc1e3b_t1):
  interview_id: '6abc1e3b-0f8a-423d-aed6-6b84640dcda1'
  turn_number: '1'
  question: '[initial mood opener]'
  anonymised_answer: 'Great team and exciting new things coming up!'
  l1_codes: '[]'


In [21]:
# ── C6: Coder (L1) ───────────────────────────────────────────────────────────
# Per interview_question_id: LLM → 1-10 open codes (noun phrases) → stored in db.

PROMPT_CODER = (
    "You are a qualitative researcher performing open coding of employee interview responses.\n"
    "Generate between {l1_min} and {l1_max} short open codes (2-5 word noun phrases) that\n"
    "capture the key conceptual ideas in the answer. Be specific and grounded in the text.\n\n"
    "--- EXAMPLE ---\n"
    "Question: How would you describe your relationship with your direct manager?\n"
    "Answer: She's always approachable and gives honest feedback. I feel trusted to make decisions.\n"
    "Output:\n"
    '{{"codes": ["managerial approachability", "honest feedback culture", "autonomy and trust"]}}\n'
    "--- END EXAMPLE ---\n\n"
    "Now code the following:\n"
    "Question: {question}\n"
    "Answer: {anonymised_answer}\n\n"
    "Return only valid JSON — no other text:\n"
    '{{"codes": ["code 1", "code 2"]}}'
)

def parse_json_safe(text: str) -> dict:
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        text  = "\n".join(lines[1:])
        if text.endswith("```"):
            text = text[:-3].strip()
    return json.loads(text)

l1_min, l1_max = L1_CODES_RANGE
for iq_id, entry in db.items():
    prompt = PROMPT_CODER.format(
        l1_min=l1_min, l1_max=l1_max,
        question=entry["question"],
        anonymised_answer=entry["anonymised_answer"]
    )
    raw    = call_llm(prompt)
    result = parse_json_safe(raw)
    db[iq_id]["l1_codes"] = result["codes"]

counts = [len(e["l1_codes"]) for e in db.values()]
print(f"L1 coding done: {len(db)} entries coded")
print(f"Codes/entry:  min={min(counts)}  max={max(counts)}  avg={sum(counts)/len(counts):.1f}")
sample_key = next(iter(db))
print(f"\nSample ({sample_key}):")
print(f"  Q: {db[sample_key]['question'][:70]}")
print(f"  Codes: {db[sample_key]['l1_codes']}")

APIConnectionError: Connection error.

In [ ]:
# ── C7: User Consolidator (L2) ───────────────────────────────────────────────
# Per interview: all L1 codes → LLM consolidates to L2_CODES_RANGE.
# LLM must return merged_from_l1 for each new code.

PROMPT_L2 = (
    "You are a qualitative researcher consolidating open codes from one employee interview into\n"
    "a unified set of between {l2_min} and {l2_max} codes. Merge overlapping or synonymous codes;\n"
    "preserve meaningfully distinct concepts. Use 2-5 word noun-phrase labels.\n"
    "You MUST list which source L1 codes each new code absorbs.\n\n"
    "--- EXAMPLE ---\n"
    'Input codes: ["managerial approachability", "manager accessibility", "open door policy", "task clarity", "clear role expectations"]\n'
    "Output:\n"
    '{{"consolidated_codes": [\n'
    '  {{"code": "accessible and open management", "merged_from_l1": ["managerial approachability", "manager accessibility", "open door policy"]}},\n'
    '  {{"code": "role and task clarity", "merged_from_l1": ["task clarity", "clear role expectations"]}}\n'
    "]}}\n"
    "--- END EXAMPLE ---\n\n"
    "All L1 codes from this interview:\n"
    "{l1_codes_list}\n\n"
    "Return only valid JSON — no other text:\n"
    '{{"consolidated_codes": [\n'
    '  {{"code": "new label", "merged_from_l1": ["source l1 code"]}},\n'
    "  ...\n"
    "]}}"
)

interview_store = {}
l2_min, l2_max  = L2_CODES_RANGE

by_interview = defaultdict(list)
for iq_id, entry in db.items():
    by_interview[entry["interview_id"]].append(entry)

for interview_id, entries in by_interview.items():
    all_l1      = [code for entry in entries for code in entry["l1_codes"]]
    l1_list_str = "\n".join(f"- {c}" for c in all_l1)
    prompt      = PROMPT_L2.format(l2_min=l2_min, l2_max=l2_max, l1_codes_list=l1_list_str)
    raw         = call_llm(prompt)
    result      = parse_json_safe(raw)
    interview_store[interview_id] = {"l2_codes": result["consolidated_codes"]}

print(f"L2 consolidation done: {len(interview_store)} interviews")
for iv_id, store in interview_store.items():
    print(f"  {iv_id[:8]}...  {len(store['l2_codes'])} L2 codes")

sample_iv = next(iter(interview_store))
print(f"\nSample L2 codes ({sample_iv[:8]}...):")
for item in interview_store[sample_iv]["l2_codes"][:3]:
    print(f"  {item['code']!r:42}  ← {item['merged_from_l1']}")

In [ ]:
# ── C8: Global Consolidator (L3) ─────────────────────────────────────────────
# All L2 codes across all interviews → LLM consolidates to L3_CODES_RANGE.
# LLM must return merged_from_l2 for each new code.

PROMPT_L3 = (
    "You are a qualitative researcher consolidating codes from {n_interviews} employee interviews\n"
    "into a final set of between {l3_min} and {l3_max} codes. Merge highly similar codes across\n"
    "interviews; keep meaningfully distinct concepts separate. Use 2-5 word noun-phrase labels.\n"
    "You MUST list which source L2 codes each new code absorbs.\n\n"
    "--- EXAMPLE ---\n"
    "Input L2 codes:\n"
    "- team and workplace culture\n"
    "- accessible and open management\n"
    "- collegial support\n"
    "- role and task clarity\n\n"
    "Output:\n"
    '{{"consolidated_codes": [\n'
    '  {{"code": "supportive management and team culture", "merged_from_l2": ["team and workplace culture", "accessible and open management", "collegial support"]}},\n'
    '  {{"code": "role clarity and expectations", "merged_from_l2": ["role and task clarity"]}}\n'
    "]}}\n"
    "--- END EXAMPLE ---\n\n"
    "All L2 codes (one per line):\n"
    "{l2_codes_list}\n\n"
    "Return only valid JSON — no other text:\n"
    '{{"consolidated_codes": [\n'
    '  {{"code": "new label", "merged_from_l2": ["source l2 code"]}},\n'
    "  ...\n"
    "]}}"
)

l3_min, l3_max = L3_CODES_RANGE
n_interviews   = len(interview_store)
all_l2_codes   = [item["code"] for store in interview_store.values() for item in store["l2_codes"]]
l2_list_str    = "\n".join(f"- {c}" for c in all_l2_codes)

prompt = PROMPT_L3.format(
    n_interviews=n_interviews, l3_min=l3_min, l3_max=l3_max,
    l2_codes_list=l2_list_str
)
raw    = call_llm(prompt)
result = parse_json_safe(raw)

global_store = {"l3_codes": result["consolidated_codes"]}

n_l3 = len(global_store["l3_codes"])
print(f"L3 consolidation done: {n_l3} codes  (range: {l3_min}-{l3_max})")
print("\nFirst 10 L3 codes:")
for item in global_store["l3_codes"][:10]:
    merged = item["merged_from_l2"]
    print(f"  {item['code']!r:42}  ← {merged}")

In [ ]:
# ── C9: Theme Clustering + Lineage ───────────────────────────────────────────
# L3 codes → LLM → clusters.  Lineage (PK = cluster name) is built in this same cell.

PROMPT_CLUSTER = (
    "You are a qualitative researcher grouping final codes into thematic clusters.\n"
    "Group into between {clusters_min} and {clusters_max} clusters.\n"
    "Each cluster must have a descriptive 3-6 word name and contain at least 2 codes.\n\n"
    "--- EXAMPLE ---\n"
    'Input L3 codes: ["supportive management culture", "open leadership style", "role clarity and expectations",\n'
    '                 "workload balance", "growth opportunities", "career development support"]\n'
    "Output:\n"
    '{{"clusters": [\n'
    '  {{"name": "Management and Leadership Style", "codes": ["supportive management culture", "open leadership style"]}},\n'
    '  {{"name": "Work Conditions and Clarity",     "codes": ["role clarity and expectations", "workload balance"]}},\n'
    '  {{"name": "Career and Growth",               "codes": ["growth opportunities", "career development support"]}}\n'
    "]}}\n"
    "--- END EXAMPLE ---\n\n"
    "Final codes (L3):\n"
    "{l3_codes_list}\n\n"
    "Return only valid JSON — no other text:\n"
    '{{"clusters": [\n'
    '  {{"name": "Cluster Name", "codes": ["l3 code 1", "l3 code 2"]}},\n'
    "  ...\n"
    "]}}"
)

clusters_min, clusters_max = CLUSTERS_RANGE
l3_list_str = "\n".join(f"- {item['code']}" for item in global_store["l3_codes"])
prompt = PROMPT_CLUSTER.format(
    clusters_min=clusters_min, clusters_max=clusters_max,
    l3_codes_list=l3_list_str
)
raw    = call_llm(prompt)
result = parse_json_safe(raw)

# ── Build lineage ─────────────────────────────────────────────────────────────
l3_to_l2 = {item["code"]: item["merged_from_l2"] for item in global_store["l3_codes"]}

l2_to_interview = {}
l2_to_l1        = {}
for interview_id, store in interview_store.items():
    for item in store["l2_codes"]:
        l2_to_interview[item["code"]] = interview_id
        l2_to_l1[item["code"]]        = item["merged_from_l1"]

l1_to_iq_ids = defaultdict(list)
for iq_id, entry in db.items():
    for code in entry["l1_codes"]:
        l1_to_iq_ids[code].append(iq_id)

lineage  = {}
clusters = {}

for cluster_def in result["clusters"]:
    name                = cluster_def["name"]
    l3_codes_in_cluster = cluster_def["codes"]

    seen_l2 = {}
    seen_l1 = {}
    for l3_code in l3_codes_in_cluster:
        for l2_code in l3_to_l2.get(l3_code, []):
            seen_l2[l2_code] = l2_to_interview.get(l2_code, "unknown")
            for l1_code in l2_to_l1.get(l2_code, []):
                if l1_code not in seen_l1 and l1_to_iq_ids[l1_code]:
                    seen_l1[l1_code] = l1_to_iq_ids[l1_code][0]

    lineage[name] = {
        "l3_codes": l3_codes_in_cluster,
        "l2_codes": [{"code": c, "interview_id": iv} for c, iv in seen_l2.items()],
        "l1_codes": [{"code": c, "interview_question_id": iq} for c, iq in seen_l1.items()],
    }
    clusters[name] = {"l3_codes": l3_codes_in_cluster, "story": ""}

print(f"Clustering done: {len(clusters)} clusters (range: {clusters_min}-{clusters_max})\n")
for name, lin in lineage.items():
    iq_ids = {e["interview_question_id"] for e in lin["l1_codes"]}
    print(f"  {name}")
    print(f"    L3: {lin['l3_codes']}")
    shown  = sorted(iq_ids)[:5]
    suffix = "..." if len(iq_ids) > 5 else ""
    print(f"    Sources: {shown}{suffix}")

In [ ]:
# ── C10: LLM Explainer ───────────────────────────────────────────────────────
# Per cluster: pull source Q+A pairs via lineage → LLM → narrative story.

PROMPT_EXPLAINER = (
    "You are a qualitative researcher writing an HR-audience narrative for a thematic cluster.\n\n"
    "Cluster: {cluster_name}\n"
    "Codes in this cluster: {codes_list}\n\n"
    "Supporting employee responses (question → anonymised answer):\n"
    "{qa_pairs_text}\n\n"
    "Write 3-5 sentences that explain what this theme means in employees' own words, "
    "highlight key patterns across respondents, and stay grounded in the evidence above.\n\n"
    "Return only the narrative text."
)

for name, lin in lineage.items():
    iq_ids = list({e["interview_question_id"] for e in lin["l1_codes"]})

    qa_lines = []
    for iq_id in iq_ids:
        if iq_id in db:
            e = db[iq_id]
            qa_lines.append(f"Q: {e['question']}\nA: {e['anonymised_answer']}")

    qa_pairs_text = "\n\n".join(qa_lines) if qa_lines else "(no source answers found)"
    codes_list    = ", ".join(clusters[name]["l3_codes"])

    prompt = PROMPT_EXPLAINER.format(
        cluster_name=name,
        codes_list=codes_list,
        qa_pairs_text=qa_pairs_text
    )
    story = call_llm(prompt)
    clusters[name]["story"] = story

    sentences = story.split(". ")
    preview   = ". ".join(sentences[:2]) + ("." if len(sentences) > 1 else "")
    print(f"\n[{name}]")
    print(f"  {preview[:150]}...")

In [ ]:
# ── C11: Persist ─────────────────────────────────────────────────────────────
# Write all data structures to OUTPUT_DIR as pretty-printed JSON.
# Re-run after any cell to checkpoint progress.

os.makedirs(OUTPUT_DIR, exist_ok=True)

to_save = {
    "db":              db,
    "interview_store": interview_store,
    "global_store":    global_store,
    "lineage":         lineage,
    "clusters":        clusters,
}

for name, data in to_save.items():
    path = os.path.join(OUTPUT_DIR, f"{name}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    size_kb = os.path.getsize(path) / 1024
    print(f"  Saved: {path}  ({size_kb:.1f} KB)")

print(f"\nAll files written to '{OUTPUT_DIR}'")

In [ ]:
# ── C12: Results display ─────────────────────────────────────────────────────
# Full lineage chain per cluster: cluster → L3 → L2 → L1 → interview_question_id → story.

print("=" * 72)
print("CODING PIPELINE — RESULTS")
print("=" * 72)

for cluster_name, lin in lineage.items():
    print(f"\n{'─' * 72}")
    print(f"CLUSTER: {cluster_name}")
    print(f"  L3 codes: {lin['l3_codes']}")
    print("  L2 codes:")
    for item in lin["l2_codes"]:
        print(f"    · {item['code']}  (interview: {item['interview_id'][:8]}...)")
    print("  L1 codes:")
    for item in lin["l1_codes"]:
        print(f"    · {item['code']}  →  {item['interview_question_id']}")
    story = clusters[cluster_name].get("story", "(not yet generated)")
    print("\n  STORY:")
    for sentence in story.replace(". ", ".\n").split("\n"):
        if sentence.strip():
            print(f"    {sentence.strip()}")